# 11e — Jeux globaux : la coordination avec information incomplète
**Navigation** : [<< 11-BayesianGames](GameTheory-11-BayesianGames-Python.ipynb) | [11c-Global-Games (sibling)](GameTheory-11c-Global-Games-Python.ipynb) | [Index](README.md)


**Autour de 11** — prolonge [11 · Jeux bayésiens](GameTheory-11-BayesianGames-Python.ipynb), [04 · Équilibres de Nash](GameTheory-04-NashEquilibrium-Python.ipynb) et le jumeau [11c — sélection d'équilibre par trembles](GameTheory-11c-Global-Games-Python.ipynb) (mêmes jeux globaux, autre question : ici l’étude du **plus petit seuil fini mesuré**, là le modèle de trembles).

Un jeu de coordination 2×2 à information **complète** porte **deux** équilibres de Nash purs (les deux accords) plus un équilibre mixte intérieur : la **multiplicité** n'est pas tranchée. Le dispositif des **jeux globaux** (Carlsson & van Damme 1993 ; Morris & Shin 1998) donne à chaque joueur un **signal bruité** d'un fondamental commun `θ`.

**La question de ce carnet** : quel **plus petit seuil fini** la recherche mesure-t-elle, et vers quel équilibre ce seuil conduit-il quand le bruit disparaît ? Le plus petit seuil fini est défini relativement à une fenêtre de recherche ; les mesures de ce carnet suggèrent une limite vers `C` lorsque le bruit décroît. Plusieurs racines de `g_sym` peuvent toutefois coexister : le carnet les compte sans conclure à une sélection globale.

> Cœur du carnet : intégration gaussienne en forme close + recherche de racine, avec une fenêtre adaptée au bruit. Le résultat est **falsifiable**.

La lecture suit une trajectoire en cinq gestes : compter les équilibres du jeu complet (section 1), regarder le canal dégradé qu'est le signal (section 2), calculer le plus petit seuil et observer sa proximité à `C` (section 3), lire les probabilités associées à ce seuil autour de `θ = C` (section 3 toujours) — et relier le tout (section 4). Les trois grandeurs affichées en route — `p*`, `k*`, P(les deux jouent H) — sont les coordonnées du même récit : le calcul compare des racines dans une classe de seuils.
**Version jumelle** : [Python](GameTheory-11e-Global-Games-Python.ipynb). Source primaire : carnet Python versionné ; bibliographie héritée sans nouvelle lecture revendiquée. Le port utilise MathNet.Numerics 5.0.0.


In [1]:
#r "nuget: MathNet.Numerics,5.0.0"
using System;
using System.Linq;
using System.Collections.Generic;
using System.Globalization;
using MathNet.Numerics.Distributions;
CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;
const double TAU2 = 1.0, C = 1.0;
// MathNet fournit les vraies densité et répartition normales.
double Phi(double x) => Normal.CDF(0, 1, x);
double PhiDensity(double x) => Normal.PDF(0, 1, x);
Console.WriteLine($"MathNet {typeof(Normal).Assembly.GetName().Version} : phi(0)={PhiDensity(0):F4}, Phi(0)={Phi(0):F4}, C={C}");


Installed Packages MathNet.Numerics, 5.0.0

MathNet 5.0.0.0 : phi(0)=0.3989, Phi(0)=0.5000, C=1


### Lecture du résultat — la forme close avec MathNet

Deux fonctions portent le calcul : la densité `PhiDensity` et la fonction de répartition `Phi` de la normale standard, fournies par MathNet.Numerics 5.0.0. Les valeurs `φ(0)=1/√(2π)` et `Φ(0)=0,5` contrôlent l'initialisation. La forme close de l'utilité de H utilise les deux fonctions, y compris le terme de densité.


## 1. Le jeu de coordination complet — la multiplicité d'origine

Deux joueurs, chacun joue **H** (investir / entrer) ou **L** (s'abstenir). Il existe un **fondamental** `θ` (par ex. la rentabilité d'un projet) : jouer H rapporte `θ` **si l'autre joue aussi H**, mais coûte `C` si l'on joue H **seul**.

| Moi \ Adverse | **H** | **L** |
|---|---|---|
| **H** | `θ` | `−C` |
| **L** | `0` | `0` |

À **information complète** (chaque joueur connaît `θ`) :
- **(H,H)** est un équilibre de Nash **si `θ ≥ 0`** (H est une meilleure réponse faible à H à l'égalité) ;
- **(L,L)** est **toujours** un équilibre (face à L, L bat H : `0 > −C`) ;
- l'**équilibre mixte** (chacun joue H avec probabilité `p* = C/(θ+C)`) existe quand `θ > 0`.

**Donc, pour `θ > 0` : deux équilibres purs + un mixte intérieur = multiplicité.** Le jeu seul ne dit pas lequel se réalisera.

Le mixte mérite un détour : il sépare deux régimes de meilleure réponse. À `θ = +0.5`, `p* = 0.6667` est la fréquence qui rend l'adversaire **indifférent** — un joueur qui mixte exactement à `p*` face au même stratège gagne zéro en espérance, comme s'il jouait L. Baissez légèrement votre probabilité : l'autre a intérêt à jouer L tout court. Augmentez-la : H domine. Le mixte sépare donc les réponses pures à une probabilité adverse donnée. Le calcul suivant cherche les seuils finis compatibles avec les signaux bruités ; il ne simule pas une dynamique d'attraction.

In [2]:
// Les gains H,H et H,L diffèrent de ceux des carnets 11c et 11f.
List<string> EquilibresPurs(double theta) =>
    theta >= 0 ? new List<string> { "HH", "LL" } : new List<string> { "LL" };
double? EquilibreMixte(double theta)
{
    double p = C / (theta + C);
    return p >= 0 && p <= 1 ? p : null;
}
foreach (double theta in new[] { .5, -.5 })
    Console.WriteLine(
        $"theta={theta:+0.0;-0.0}, purs={string.Join(",", EquilibresPurs(theta))}, mixte={EquilibreMixte(theta)?.ToString("F4") ?? "aucun"}"
    );


theta=+0.5, purs=HH,LL, mixte=0.6667


theta=-0.5, purs=LL, mixte=aucun


### Lecture — la multiplicité en nombres

La sortie sépare deux mondes en un seul paramètre. À `θ = +0.5`, le jeu porte **deux** équilibres purs, `('H', 'H')` et `('L', 'L')`, plus un mixte `p* = 0.6667` : chacun investit avec deux chances sur trois, et c'est exactement la fréquence qui rend l'autre indifférent — à `p*`, jouer H et jouer L rapportent tous deux zéro en espérance. À `θ = −0.5`, la liste ne contient plus que `('L', 'L')` : le gain commun d'un investissement réussi est négatif, H n'est jamais une meilleure réponse, et le mixte disparaît car `p* = C/(θ+C)` déborderait de `[0, 1]` — la fonction le signale en renvoyant `null`.

Retenez la mécanique du mixte : l'indifférence exige `p·θ − (1 − p)·C = 0`, d'où `p* = C/(θ+C)`. Quand `θ` tend vers 0 par valeurs positives, `p*` tend vers 1 — il faut être presque sûr que l'autre investit pour oser investir dans un projet à peine rentable. C'est cette fragilité de l'accord ('H', 'H'), qui ne survit que par certitude mutuelle, que le bruit va exploiter.

## 2. Le signal bruité et la stratégie à seuil

Personne ne connaît `θ` directement. Chaque joueur `i` observe un **signal bruité**

$$s_i = \theta + \varepsilon_i, \qquad \varepsilon_i \sim \mathcal{N}(0, \sigma^2) \ \text{(indépendants).}$$

On restreint aux **stratégies à seuil** : jouer **H** si et seulement si `s_i ≥ k`. La décision se résume à une **frontière** `k` dans l'espace des signaux — la classe de stratégies naturelle quand l'information est un signal du fondamental.

Économiquement, `s = θ + ε` se lit : chaque joueur observe le fondamental à travers son propre canal dégradé — deux analystes voient la même économie avec des modèles et des données légèrement différents. La variance `σ²` mesure la qualité du canal ; restreindre aux stratégies à seuil dit seulement que la décision finale est binaire : au-dessus de `k`, le signal paraît assez bon pour tenter l'investissement. Toute la subtilité du jeu passera dans le choix de cette frontière.

In [3]:
// Six signaux privés pour un état fixé, avec le générateur normal MathNet.
var rng = new Random(42);
double theta0 = .5,
    sigma0 = .3;
var sample = Enumerable.Range(0, 6).Select(_ => theta0 + Normal.Sample(rng, 0, sigma0)).ToArray();
Console.WriteLine(
    $"theta={theta0}, sigma={sigma0} : {string.Join(", ", sample.Select(x => x.ToString("F3")))}"
);


theta=0.5, sigma=0.3 : 0.623, 0.179, 0.280, 1.036, 0.302, 0.246


### Lecture — ce que chaque joueur voit

Pour le même fondamental `θ=0,5` et `σ=0,3`, les six signaux de cette exécution s'étalent de `0,179` à `1,036` : tous sont positifs dans cette réalisation, mais le plus haut dépasse deux fois le fondamental et le plus bas lui est nettement inférieur. Le tirage varie avec le générateur ; les signaux sont indépendants conditionnellement à `θ`.

Le canal bruité donne ainsi des observations différentes d'une même économie. Chaque joueur raisonne sur le signal de l'autre sans le connaître : le modèle combine cette incertitude avec la mise à jour gaussienne du fondamental. Le seuil `k` est la frontière au-dessus de laquelle jouer H. La question suivante cherche les seuils compatibles avec ces croyances, puis distingue la plus petite racine finie mesurée des autres racines.


## 3. Le plus petit seuil fini mesuré

Avec un **a priori** `θ ~ N(0, τ²)` et un signal gaussien, la croyance a posteriori d'un joueur de signal `s` est gaussienne : `θ | s ~ N(m, v)` avec

$$m = \tfrac{v\,s}{\sigma^2}, \qquad v = \Big(\tfrac{1}{\tau^2} + \tfrac{1}{\sigma^2}\Big)^{-1}.$$

Face à un adversaire de seuil `k`, l'utilité espérée de **H** s'écrit **en forme close** (intégration gaussienne) :

$$\mathbb{E}[U_H \mid s] = (m+C)\,\Phi\!\Big(\tfrac{m-k}{w}\Big) + \tfrac{v}{w}\,\varphi\!\Big(\tfrac{m-k}{w}\Big) - C, \qquad w = \sqrt{v + \sigma^2},$$

où `Φ` et `φ` sont la fonction de répartition et la densité de la normale standard. Un **seuil d'équilibre** `k*` est une valeur où un joueur de signal `s = k*` est **indifférent** face à un adversaire de même seuil : c'est une racine de `g(k) = U_H(k; k) = 0`. Le **plus petit seuil fini trouvé dans la fenêtre** est étudié ici ; les autres racines et les équilibres hors de cette classe ne sont pas écartés.

**Le point clé** : dans les fenêtres et bruits étudiés, la plus petite racine finie trouvée se rapproche du **coût d'agir seul** `C`. Cela ne prouve **pas** qu'il est la seule racine : la mesure suivante porte sur les racines de la fenêtre et non sur tous les équilibres.

D'où viennent `m` et `v` ? De la règle de Bayes pour deux gaussiennes : l'a priori `θ ~ N(0, τ²)` et le signal `s` se combinent en une croyance `θ | s ~ N(m, v)` dont les **précisions** s'additionnent (`1/v = 1/τ² + 1/σ²`) et dont la moyenne penche vers le signal à raison de sa précision relative. L'utilité de H en découle par intégration gaussienne — la probabilité que l'autre reçoive un signal au-dessus de `k` sachant le mien — et tout se résout en la forme close affichée : l'équilibre entier tient dans deux fonctions spéciales, `Φ` et `φ`.

In [4]:
// La forme close inclut le terme de densité : l'état et l'entrée adverse sont corrélés.
(double Mean, double Variance) Post(double signal, double sigma)
{
    double variance = 1 / (1 / TAU2 + 1 / (sigma * sigma));
    return (variance * signal / (sigma * sigma), variance);
}
double UtiliteH(double signal, double cutoff, double sigma)
{
    var (mean, variance) = Post(signal, sigma);
    double w = Math.Sqrt(variance + sigma * sigma);
    double z = (mean - cutoff) / w;
    return (mean + C) * Phi(z) + (variance / w) * PhiDensity(z) - C;
}
double GSym(double k, double sigma) => UtiliteH(k, k, sigma);

// Balayage de 2000 points puis 80 bissections : première racine finie dans [0,kMax].
double SeuilEquilibre(double sigma, double kMax = 30, int n = 2000)
{
    double g0 = GSym(0, sigma);
    if (g0 >= 0)
        return 0;
    for (int i = 0; i < n - 1; i++)
    {
        double a = kMax * i / (n - 1),
            b = kMax * (i + 1) / (n - 1),
            ga = GSym(a, sigma);
        if (ga * GSym(b, sigma) <= 0)
        {
            for (int j = 0; j < 80; j++)
            {
                double mid = (a + b) / 2;
                if (ga * GSym(mid, sigma) <= 0)
                    b = mid;
                else
                {
                    a = mid;
                    ga = GSym(mid, sigma);
                }
            }
            return (a + b) / 2;
        }
    }
    return double.NaN;
}

// 40000 intervalles dans la fenêtre [0,30/sigma] sauf borne demandée.
int CompteRacines(double sigma, double? kMax = null, int n = 40000)
{
    double limit = kMax ?? 30 / sigma,
        step = limit / n,
        previous = GSym(0, sigma);
    int count = 0;
    for (int i = 1; i <= n; i++)
    {
        double current = GSym(i * step, sigma);
        if (previous * current < 0)
            count++;
        previous = current;
    }
    return count;
}
Console.WriteLine($"sigma=0.3 : plus petit seuil fini dans [0,30] = {SeuilEquilibre(.3):F4}");


sigma=0.3 : plus petit seuil fini dans [0,30] = 1.5908


### Lecture — pourquoi k* dépasse C

À `σ = 0.3`, le plus petit seuil d'équilibre vaut `k* = 1.5908`, nettement au-dessus de `C = 1`. La lecture naïve — « le seuil devrait se caler sur le coût d'agir seul » — est fausse tant que le bruit est fini : l'écart `k* − C ≈ 0.59` mesure la **double incertitude** d'un joueur. Il doute d'abord sur le fondamental (il ne voit que `s`, pas `θ`), puis sur le signal de l'autre (même à fondamental donné, l'adversaire observe un `s'` différent du sien). Pour oser jouer H, il faut être optimiste sur `θ` **et** sur la probabilité que l'autre reçoive aussi un signal au-dessus du seuil — deux sources d'incertitude prises en compte par la même espérance conditionnelle, avec une racine mesurée au-dessus du cas sans bruit.

La forme close de l'indifférence symétrique encode ces croyances. Sur cette branche, la première racine est une traversée du négatif vers le positif ; une seconde traversée existe dans la fenêtre élargie. Le balayage localise la première racine et le comptage conserve les autres traversées au lieu de les confondre avec une sélection globale.


### Exercice 1 — probabilité côté L

Pour un état donné et plusieurs bruits, renvoyez une probabilité par bruit que les deux joueurs choisissent L. Les signaux sont indépendants sachant l’état : calculez la probabilité individuelle à partir de la CDF normale et du seuil fini mesuré, puis combinez-les.


In [5]:
// Exercice 1 : probabilité que les deux agents jouent L.
Func<double, double[], double[]> probBothL = (theta, sigmas) =>
{
    // TODO : compléter pour chaque bruit.
    return null;
};
var cases1 = new[]
{
    (-.5, new[] { .3, .1, .01 }),
    (1.2, new[] { .3, .2 }),
    (.8, new[] { .1, .05 }),
};
var answers1 = cases1.Select(x => probBothL(x.Item1, (double[])x.Item2.Clone())).ToArray();
if (answers1.All(x => x == null))
    Console.WriteLine("Exercice 1 à compléter");
else
{
    for (int i = 0; i < cases1.Length; i++)
    {
        var (theta, sigmas) = cases1[i];
        var ans = answers1[i];
        if (ans == null || ans.Length != sigmas.Length)
            throw new Exception("Réponse partielle");
        for (int j = 0; j < sigmas.Length; j++)
        {
            double p = 1 - Phi((theta - SeuilEquilibre(sigmas[j])) / sigmas[j]);
            if (!double.IsFinite(ans[j]) || Math.Abs(ans[j] - p * p) > 1e-9)
                throw new Exception("Probabilité incorrecte");
        }
    }
    Console.WriteLine("Exercice 1 : probabilités vérifiées");
}


Exercice 1 à compléter


In [6]:
// Les six fenêtres adaptatives gardent chacune les 40000 intervalles.
Console.WriteLine("sigma | nombre de racines sur [0,30/sigma] | première racine sur [0,30]");
foreach (double s in new[] { .3, .2, .1, .05, .01, .001 })
    Console.WriteLine($"{s:F3} | {CompteRacines(s)} | {SeuilEquilibre(s):F4}");


sigma | nombre de racines sur [0,30/sigma] | première racine sur [0,30]


0.300 | 2 | 1.5908


0.200 | 2 | 1.2334


0.100 | 2 | 1.0799


0.050 | 2 | 1.0335


0.010 | 2 | 1.0058


0.001 | 2 | 1.0006


### Lecture — deux racines dans les fenêtres mesurées

Deux régularités traversent le tableau. D'abord le **nombre de racines : 2** à chaque niveau de bruit — la multiplicité n'est pas éliminée par le bruit, elle est réorganisée. Ensuite la **dérive du plus petit seuil** : `k*` descend de `1.5908` (σ = 0.3) à `1.0006` (σ = 0.001), et la cible de la convergence est `C = 1`, pas `0`. Quand l'observation devient presque parfaite, le seuil d'équilibre se fixe sur le **coût d'agir seul** — la frontière de dominance de risque de ce jeu symétrique complet : le mixte d'indifférence `p*=C/(θ+C)` vaut `1/2` en `θ=C`.

Le terme « robuste » est une appellation du carnet source pour la plus petite racine finie mesurée. Le balayage ne démontre ni résistance aux perturbations des croyances, ni sélection au sens de Carlsson et van Damme. Il exclut les équilibres hors de la classe de seuils et ne dit rien hors de la fenêtre.


### Exercice 2 — seuil et bruit

Renvoyez un seuil fini mesuré par bruit avec `SeuilEquilibre`, pour étudier sa proximité à `C`. Lorsque la recherche ne trouve aucune racine finie sur `[0,30]`, conservez `double.NaN`.


In [7]:
// Exercice 2 : les seuils finis pour une liste de bruits.
Func<double[], double[]> seuils = sigmas =>
{
    // TODO : retourner un seuil par bruit.
    return null;
};
var cases2 = new[] { new[] { 1.0, .3, .05, .001 }, new[] { .2, .1 } };
var answers2 = cases2.Select(x => seuils((double[])x.Clone())).ToArray();
if (answers2.All(x => x == null))
    Console.WriteLine("Exercice 2 à compléter");
else
{
    for (int i = 0; i < cases2.Length; i++)
    {
        if (answers2[i] == null || answers2[i].Length != cases2[i].Length)
            throw new Exception("Réponse partielle");
        for (int j = 0; j < cases2[i].Length; j++)
        {
            double expected = SeuilEquilibre(cases2[i][j]);
            double value = answers2[i][j];
            if (double.IsNaN(expected) ? !double.IsNaN(value) : !double.IsFinite(value) || Math.Abs(value - expected) > 1e-9)
                throw new Exception("Seuil incorrect");
        }
    }
    Console.WriteLine("Exercice 2 : seuils vérifiés");
}


Exercice 2 à compléter


In [8]:
// Probabilité conditionnelle d'action H pour un fondamental fixé.
double theta = 1.2,
    sigma = .3,
    kstar = SeuilEquilibre(sigma),
    pH = Phi((theta - kstar) / sigma);
Console.WriteLine($"theta={theta}, sigma={sigma}, seuil={kstar:F4}, P(H)={pH:F4}");


theta=1.2, sigma=0.3, seuil=1.5908, P(H)=0.0964


### Probabilités sur la plus petite racine finie

Pour le plus petit seuil trouvé dans la fenêtre, la part conditionnelle de joueurs qui jouent H est `P(H) = Φ((θ − k*)/σ)` — une valeur **interne** (ni 0 ni 1). D'autres racines peuvent exister ; cette mesure ne prouve pas leur sélection ou leur rejet.

**Branche bornée à petit bruit.** Pour `k` borné, la formule donne `m→k`, `v/w→0` et `(m-k)/w→0`, donc `g_sym(k,σ)→(k-C)/2`. Une branche de racines qui admet une limite finie doit donc tendre vers `C` ; cela ne décrit pas les racines qui s'échappent de la fenêtre. Sur cette branche, les signaux convergent vers `θ`, donc :

- si **`θ > C`** : `P(les deux jouent H) → 1`  →  la probabilité de (H,H) tend vers 1 sur cette branche ;
- si **`θ < C`** : `P(les deux jouent H) → 0`  →  la probabilité de (H,H) tend vers 0 sur cette branche — et c'est un **échec de coordination** dès que `0 < θ < C` (l'équilibre efficace (H,H) existait, cette branche n'y conduit pas).

Dans les valeurs calculées, la transition de cette branche se rapproche de `C`. La fenêtre et la classe de seuils ne suffisent pas à démontrer une sélection robuste par perturbations.

La borne `θ = C` est un énoncé de limite. À bruit fini, les probabilités ne sont pas encore extrêmes : les colonnes de la cellule suivante montrent le phénomène en train de se faire, du « presque jamais » à bruit fort au « presque toujours » à bruit faible — et la transition entre les deux est raide. C'est cette raideur, mesurée, qui fait des jeux globaux un instrument prédictif : la frontière est fine et calculable.

In [9]:
// Le carré utilise l'indépendance des signaux sachant theta.
double ProbBothH(double theta, double sigma)
{
    double p = Phi((theta - SeuilEquilibre(sigma)) / sigma);
    return p * p;
}
foreach (double t in new[] { 1.2, .8 })
{
    Console.WriteLine($"theta={t:F1}");
    foreach (double s in new[] { .3, .2, .1, .05, .01, .001 })
        Console.WriteLine($"sigma={s:F3} seuil={SeuilEquilibre(s):F4} P(H,H)={ProbBothH(t, s):F4}");
}


theta=1.2


sigma=0.300 seuil=1.5908 P(H,H)=0.0093


sigma=0.200 seuil=1.2334 P(H,H)=0.1880


sigma=0.100 seuil=1.0799 P(H,H)=0.7836


sigma=0.050 seuil=1.0335 P(H,H)=0.9991


sigma=0.010 seuil=1.0058 P(H,H)=1.0000


... (4 outputs left out)

sigma=0.100 seuil=1.0799 P(H,H)=0.0000


sigma=0.050 seuil=1.0335 P(H,H)=0.0000


sigma=0.010 seuil=1.0058 P(H,H)=0.0000


sigma=0.001 seuil=1.0006 P(H,H)=0.0000


### Lecture — la transition sur la branche mesurée

Pour `θ = 1.2` (au-dessus de `C`), la coordination sur (H, H) grimpe de `0.0093` à `σ = 0.3` — le seuil `k* = 1.5908` reste hors de portée de la plupart des signaux — puis perce : `0.1880` à `σ = 0.2`, `0.7836` à `σ = 0.1`, `0.9991` à `σ = 0.05`, et `1.0000` dès `σ = 0.01`. Pour `θ = 0.8` (sous `C`), la colonne reste collée à zéro, `0.0002` au maximum à `σ = 0.2` : sur un projet dont la rentabilité ne couvre pas le coût d'agir seul, la coordination sur (H,H) reste très peu probable pour les bruits de la table.

La transition de la première colonne est raide : entre `σ = 0.2` et `σ = 0.05`, un facteur 4 de bruit fait passer la probabilité d'un cinquième à la quasi-certitude. C'est le **mur** — la transition est raide. Et son point de bascule n'est pas `θ = 0` (apparition de l'accord pur (H,H) dans le jeu complet) mais `θ = C`. Entre les deux s'ouvre la zone des échecs de coordination : `0 < θ < C` rendrait (H, H) efficient à information complète, et la branche de seuils mesurée rend (H,H) peu probable au profit de (L,L) — la crise de change qui n'a pas lieu parce que chacun doute que l'autre attaque.
Ces probabilités portent sur la plus petite racine finie trouvée dans la fenêtre, et ne prouvent pas une sélection par perturbations ni l’absence d’équilibres hors classe.


### Exercice 3 — racine dans une fenêtre

Renvoyez si `CompteRacines` trouve exactement une racine sur la fenêtre demandée. Sans borne explicite, utilisez `[0,30/sigma]` ; le résultat ne porte pas sur les racines hors de cette fenêtre.


In [10]:
// Exercice 3 : unicité uniquement dans la fenêtre demandée.
Func<double, double?, bool?> estUnique = (sigma, kMax) =>
{
    // TODO : compter les racines dans la fenêtre.
    return null;
};
var cases3 = new (double, double?)[] { (.05, null), (.3, 3.0), (.3, 1.0) };
var answers3 = cases3.Select(x => estUnique(x.Item1, x.Item2)).ToArray();
if (answers3.All(x => x == null))
    Console.WriteLine("Exercice 3 à compléter");
else
{
    for (int i = 0; i < cases3.Length; i++)
        if (
            answers3[i] == null
            || answers3[i] != (CompteRacines(cases3[i].Item1, cases3[i].Item2) == 1)
        )
            throw new Exception("Comptage incorrect ou réponse partielle");
    Console.WriteLine("Exercice 3 : fenêtres vérifiées");
}


Exercice 3 à compléter


## 4. Synthèse

| Information | Équilibres étudiés | Résultat quand `σ → 0` |
|---|---|---|
| **Complète** (`θ` connu) | **multiplicité** : (H,H), (L,L), mixte | — |
| **Incomplète** (signal `s = θ + ε`) | **plus petit seuil fini mesuré** `k*` ; autres racines possibles | probabilité de (H,H) sur la branche mesurée tendant vers 1 si `θ > C`, vers 0 si `θ < C` |

La fenêtre et le calcul à seuil donnent une branche mesurée, sans démontrer une sélection par perturbations. C'est ce qui rend les jeux globaux utiles pour étudier les crises de change, les paniques bancaires et la coordination sur un fondamental.

Les références héritées du Python citent Morris & Shin (1998) comme application aux attaques de change. Les gains H/H et H/L de ce carnet en donnent une analogie pédagogique : une action collective peut réussir alors qu'une action isolée coûte C. Le calcul présent fait varier la précision des **signaux privés** ; il ne mesure ni une communication publique ni l'effet d'une intervention de banque centrale. Les probabilités rapportées portent sur la branche de seuils définie ci-dessus.


### Synthèse des exercices

Les trois exercices réutilisent la probabilité de coordination, les seuils et le comptage des racines dans une fenêtre. Les contrôles portent sur les données mesurées et ne démontrent pas une sélection au-delà de ces fenêtres.


## Références

- **Carlsson, H., & van Damme, E. (1993).** « Global Games and Equilibrium Selection », *Econometrica* 61(5), 989–1018. — introduction du jeu global et de la sélection d'équilibre sous information bruitée.
- **Morris, S., & Shin, H. S. (1998).** « Unique Equilibrium in a Model of Self-Fulfilling Currency Attacks », *American Economic Review* 88(3), 587–597. — application canonique aux attaques de change.

**Place dans la série** : `11e`, « Autour de 11 » — avec `11` (jeux bayésiens), `11b` (Lean), `11c` (trembles) et `11d` (raffinements séquentiels).
